In [ ]:
'''Imports.'''

%load_ext autoreload
%autoreload 2

from utils.config import TEMP_SOLD_DIR
from utils.io import read_df_from_path, read_multiple_df
from stats.filtering import (
    FilterConfig,
    drop_empty_records, remove_trailing_name_text, 
    keep_records_containing, remove_records_containing, 
    remove_trailing_name_text, conv_dates, set_date_as_index,
    replace_null_shipping_with_avg
    )
from stats.visualisation import show_price_hist
from stats.calculations import fit_to_lognormal, fit_to_normal, calculate_simple_stats
from stats.stat_filtering import filter_by_percentile, filter_by_price, filter_by_standard_deviation
from stats.generate_pdf import pdf_normal, pdf_lognormal

In [ ]:
'''Load and view raw data.'''

csv_path = TEMP_SOLD_DIR / "iphone 16"
raw_df = read_multiple_df(csv_path) #reads all df for a product at once

raw_df.head()

In [ ]:
'''Filter data'''

filter_conf = FilterConfig(
    required_txts = ['iphone 16'],
    banned_txts = ['pro max', "pro", "plus"]
)


filtered_df = (
    raw_df
    .pipe(drop_empty_records)
    .pipe(remove_trailing_name_text)
    .pipe(keep_records_containing, filter_conf)
    .pipe(remove_records_containing, filter_conf)
    .pipe(conv_dates)
    .pipe(set_date_as_index)
    .pipe(replace_null_shipping_with_avg)
)

# TODO: Get data from last N days
filtered_df.sort_values(by='sold_date')

In [ ]:
'''Looking at specific records.'''

filtered_df.head().columns

In [ ]:
'''Visualising data.'''

show_price_hist(filtered_df)

In [ ]:
#data seems bimodal, multiple items present
#fixed by fitering out iphone pros as well, now fits lognormal distribution

In [ ]:
'''Clean data using std'''

cleaned_df = filter_by_standard_deviation(filtered_df, n=2)

show_price_hist(cleaned_df)

In [ ]:
simple_stats = calculate_simple_stats(cleaned_df)
normal_params = fit_to_normal(cleaned_df)
lognormal_params = fit_to_lognormal(cleaned_df)

max = simple_stats["max"]
min = simple_stats["min"]

normal_pdf = pdf_normal(normal_params, max=max, min=min)
lognormal_pdf = pdf_lognormal(lognormal_params, max=max, min=min)

show_price_hist(cleaned_df, dist_values=lognormal_pdf, lines=simple_stats, bin_size=20)
